# Day 6 — SQL Practice Questions: Window Functions Part 1

**3 questions** · Easy → Medium  
Tables created inline — no DBeaver setup required.

In [ ]:
%load_ext sql

USERNAME = 'postgres'
PASSWORD = 'hariom'
HOST     = 'localhost'
PORT     = '5432'
DBNAME   = 'postgres'

%sql postgresql://{USERNAME}:{PASSWORD}@{HOST}:{PORT}/{DBNAME}

In [ ]:
%%sql
DROP TABLE IF EXISTS d6w_sales CASCADE;

CREATE TABLE d6w_sales (
    sale_id  SERIAL PRIMARY KEY,
    emp_id   INTEGER,
    emp_name VARCHAR(20),
    region   VARCHAR(10),
    month    INTEGER,
    revenue  NUMERIC(10, 2)
);

INSERT INTO d6w_sales (emp_id, emp_name, region, month, revenue) VALUES
  (1, 'Alice',  'North',  1, 5000.00),
  (2, 'Bob',    'North',  1, 4200.00),
  (3, 'Carol',  'North',  1, 4200.00),
  (4, 'Dave',   'North',  1, 3800.00),
  (5, 'Eve',    'South',  1, 6100.00),
  (6, 'Frank',  'South',  1, 5500.00),
  (7, 'Grace',  'South',  1, 5500.00),
  (8, 'Hank',   'South',  1, 4000.00),
  (1, 'Alice',  'North',  2, 5300.00),
  (2, 'Bob',    'North',  2, 4800.00),
  (3, 'Carol',  'North',  2, 4000.00),
  (5, 'Eve',    'South',  2, 5900.00),
  (6, 'Frank',  'South',  2, 6300.00),
  (7, 'Grace',  'South',  2, 5100.00);

SELECT 'Tables created' AS status;

In [ ]:
%%sql
SELECT * FROM d6w_sales ORDER BY region, month, revenue DESC;

---
## Q1 (Easy) — Rank Employees by Revenue Within Each Region

For every row in `d6w_sales`, add a `revenue_rank` column that ranks employees by revenue **within each region** (all months combined).  
Use `RANK()`. Order the final output by `region`, `revenue_rank`.

**Expected columns:** `emp_id`, `emp_name`, `region`, `month`, `revenue`, `revenue_rank`

**Warm-ups:**
- What is the OVER clause syntax for "partition by region, order by revenue descending"?
- What rank does Carol get in North if Bob and Carol both have the same revenue?
- What happens to the next rank number after a tie with RANK()?

In [ ]:
%%sql
-- YOUR QUERY HERE
-- Use RANK() OVER (PARTITION BY region ORDER BY revenue DESC)

### Solution

In [ ]:
%%sql
-- SOLUTION
SELECT emp_id,
       emp_name,
       region,
       month,
       revenue,
       RANK() OVER (PARTITION BY region ORDER BY revenue DESC) AS revenue_rank
FROM   d6w_sales
ORDER  BY region, revenue_rank;
-- Note: PARTITION BY region only (not region+month) → ranks across all months in a region

---
## Q2 (Medium) — Top 2 Employees per Region (Ties Handled Correctly)

Find the top 2 employees by revenue **within each region** for **month = 1** only.  
If two employees are tied at rank 2, both should appear.

Use `DENSE_RANK` + a CTE to filter.

**Expected:**  
- North: Alice(1), Bob(2), Carol(2) — 3 rows (tie at rank 2)  
- South: Eve(1), Frank(2), Grace(2) — 3 rows (tie at rank 2)

**Warm-ups:**
- Why DENSE_RANK and not RANK for top-N filtering?
- Why can't you write `WHERE DENSE_RANK() OVER (...) <= 2` directly?
- What is the difference in result between `<= 2` with RANK vs DENSE_RANK when there are ties?

In [ ]:
%%sql
-- YOUR QUERY HERE
-- Use a CTE: WITH ranked AS (...) SELECT * FROM ranked WHERE drnk <= 2

### Solution

In [ ]:
%%sql
-- SOLUTION
WITH ranked AS (
    SELECT emp_id,
           emp_name,
           region,
           month,
           revenue,
           DENSE_RANK() OVER (PARTITION BY region ORDER BY revenue DESC) AS drnk
    FROM   d6w_sales
    WHERE  month = 1
)
SELECT * FROM ranked
WHERE  drnk <= 2
ORDER  BY region, drnk;
-- 6 rows: 3 per region (because of ties at rank 2)

---
## Q3 (Medium) — Employees Who Ranked #1 in Any Region for Any Month

Find every `(emp_id, emp_name, region, month)` where the employee was the **top revenue earner** in their region that month.  
Partition by `(region, month)`, use `RANK()`. Return only rows where rank = 1.

**Expected results:**  
- Alice: North/month1 (5000), North/month2 (5300)  
- Eve: South/month1 (6100)  
- Frank: South/month2 (6300)

**Warm-ups:**
- How do you partition by TWO columns?
- If two employees tied for #1 in the same region+month, would both appear? Why?
- Should you use DISTINCT here? Why or why not?

In [ ]:
%%sql
-- YOUR QUERY HERE
-- PARTITION BY region, month ORDER BY revenue DESC
-- Filter to rank = 1

### Solution

In [ ]:
%%sql
-- SOLUTION
WITH ranked AS (
    SELECT emp_id,
           emp_name,
           region,
           month,
           revenue,
           RANK() OVER (PARTITION BY region, month ORDER BY revenue DESC) AS rnk
    FROM   d6w_sales
)
SELECT emp_id, emp_name, region, month, revenue
FROM   ranked
WHERE  rnk = 1
ORDER  BY region, month;
-- Alice tops North in both months
-- Eve tops South/month1, Frank tops South/month2